# Module 05: HuggingFace Accelerate

Read `README.md` before starting.

**Goal**: One training script, three backends (DDP / FSDP / DeepSpeed), zero code changes.

In [ ]:
!pip install -q accelerate==0.34.0 transformers==4.44.0 datasets==2.21.0 deepspeed==0.14.0

import torch
assert torch.cuda.device_count() == 2
print(f"Ready. {torch.cuda.device_count()} GPUs.")

## Part 1: Write the Accelerate Config Files

We write three configs — one per strategy. The training script will not change between them.

In [ ]:
import yaml
import json

# ─── Config 1: DDP ────────────────────────────────────────────────────────────
ddp_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "MULTI_GPU",
    "num_processes": 2,
    "gpu_ids": "all",
    "mixed_precision": "fp16",
    "machine_rank": 0,
    "main_process_ip": None,
    "main_process_port": None,
    "rdzv_backend": "static",
    "same_network": True,
    "use_cpu": False,
}
with open("/kaggle/working/accel_ddp.yaml", "w") as f:
    yaml.dump(ddp_config, f)

# ─── Config 2: FSDP ZeRO-3 ────────────────────────────────────────────────────
fsdp_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "FSDP",
    "num_processes": 2,
    "mixed_precision": "fp16",
    "fsdp_config": {
        "fsdp_auto_wrap_policy": "TRANSFORMER_BASED_WRAP",
        "fsdp_backward_prefetch_policy": "BACKWARD_PRE",
        "fsdp_cpu_ram_efficient_loading": True,
        "fsdp_forward_prefetch": False,
        "fsdp_offload_params": False,
        "fsdp_sharding_strategy": "FULL_SHARD",
        "fsdp_state_dict_type": "FULL_STATE_DICT",
        "fsdp_sync_module_states": True,
        "fsdp_use_orig_params": True,
    },
}
with open("/kaggle/working/accel_fsdp.yaml", "w") as f:
    yaml.dump(fsdp_config, f)

# ─── Config 3: DeepSpeed ZeRO-3 ───────────────────────────────────────────────
ds_zero3 = {
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "fp16": {"enabled": "auto"},
    "zero_optimization": {
        "stage": 3,
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "reduce_scatter": True,
        "reduce_bucket_size": "auto",
        "overlap_comm": True,
        "contiguous_gradients": True,
        "stage3_param_persistence_threshold": "auto",
        "stage3_max_live_parameters": 1e9,
    },
}
with open("/kaggle/working/ds_zero3.json", "w") as f:
    json.dump(ds_zero3, f, indent=2)

deepspeed_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "DEEPSPEED",
    "num_processes": 2,
    "mixed_precision": "fp16",
    "deepspeed_config": {
        "deepspeed_config_file": "/kaggle/working/ds_zero3.json",
        "zero3_init_flag": True,
    },
}
with open("/kaggle/working/accel_deepspeed.yaml", "w") as f:
    yaml.dump(deepspeed_config, f)

print("Config files written:")
print("  /kaggle/working/accel_ddp.yaml")
print("  /kaggle/working/accel_fsdp.yaml")
print("  /kaggle/working/accel_deepspeed.yaml")

## Part 2: The Unified Training Script

This script does NOT import `dist`, does NOT call `init_process_group`,
does NOT use `DistributedSampler` directly, and does NOT know what backend it is using.
Accelerate handles all of that.

In [ ]:
%%writefile /kaggle/working/accel_train.py
"""
Unified training script using HuggingFace Accelerate.

Works with any of:
  accelerate launch --config_file accel_ddp.yaml       accel_train.py
  accelerate launch --config_file accel_fsdp.yaml      accel_train.py
  accelerate launch --config_file accel_deepspeed.yaml accel_train.py
"""
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from accelerate import Accelerator
from accelerate.utils import set_seed

# ─── Config ───────────────────────────────────────────────────────────────────
BATCH_SIZE = 8
SEQ_LEN = 128
VOCAB_SIZE = 10000
D_MODEL = 256
NUM_LAYERS = 6
LR = 1e-4
NUM_EPOCHS = 3
DATASET_SIZE = 800
SEED = 42

# ─── Model ────────────────────────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                D_MODEL, nhead=8, dim_feedforward=D_MODEL*4,
                dropout=0.1, batch_first=True
            )
            for _ in range(NUM_LAYERS)
        ])
        self.head = nn.Linear(D_MODEL, 2)

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(x[:, 0, :])


def main():
    set_seed(SEED)

    # ── Step 1: Create the Accelerator ────────────────────────────────────────
    # It reads the config from the environment (set by `accelerate launch`)
    # You can also pass mixed_precision="fp16" directly here
    accelerator = Accelerator()

    accelerator.print(f"Backend: {accelerator.distributed_type}")
    accelerator.print(f"Num processes: {accelerator.num_processes}")
    accelerator.print(f"Mixed precision: {accelerator.mixed_precision}")
    accelerator.print(f"Device: {accelerator.device}")

    # ── Step 2: Build model and optimizer as normal ────────────────────────────
    model = TransformerClassifier()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    # ── Step 3: Build DataLoader — no DistributedSampler needed ───────────────
    # Accelerate wraps the dataloader and handles sharding automatically
    dataset = TensorDataset(
        torch.randint(0, VOCAB_SIZE, (DATASET_SIZE, SEQ_LEN)),
        torch.randint(0, 2, (DATASET_SIZE,))
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    # ── Step 4: Prepare — this one call does everything ───────────────────────
    # Depending on the config it will:
    #   - Wrap model with DDP / FSDP / DeepSpeed
    #   - Move model to the right device
    #   - Add DistributedSampler to the dataloader
    #   - Set up mixed precision
    model, optimizer, loader = accelerator.prepare(model, optimizer, loader)

    # ── Step 5: Training loop — almost identical to single-GPU ────────────────
    throughputs = []
    step = 0

    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.0

        for x, y in loader:
            t0 = time.perf_counter()

            # No need to move data to device — accelerator handles it
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)

            # Use accelerator.backward() instead of loss.backward()
            # This handles gradient scaling for fp16 and DeepSpeed backward
            accelerator.backward(loss)

            optimizer.step()

            elapsed = time.perf_counter() - t0
            tp = (BATCH_SIZE * accelerator.num_processes) / elapsed
            throughputs.append(tp)
            epoch_loss += loss.item()
            step += 1

        # Only print from main process
        if accelerator.is_main_process:
            avg_loss = epoch_loss / len(loader)
            avg_tp = sum(throughputs[-len(loader):]) / len(loader)
            print(f"[Epoch {epoch}] loss={avg_loss:.4f} | throughput={avg_tp:.0f} samples/s")

    # ── Step 6: Save checkpoint via Accelerate ────────────────────────────────
    # This handles FSDP/DeepSpeed state gathering automatically
    accelerator.wait_for_everyone()  # Sync all processes before saving
    unwrapped = accelerator.unwrap_model(model)  # Get the underlying nn.Module
    accelerator.save(
        unwrapped.state_dict(),
        "/kaggle/working/accel_checkpoint.pt"
    )

    if accelerator.is_main_process:
        total_tp = sum(throughputs) / len(throughputs)
        print(f"\nDone. Avg throughput: {total_tp:.0f} samples/s")
        print("Checkpoint saved to /kaggle/working/accel_checkpoint.pt")


if __name__ == "__main__":
    main()

In [ ]:
print("=" * 60)
print("Run 1: DDP backend")
print("=" * 60)
!accelerate launch --config_file /kaggle/working/accel_ddp.yaml /kaggle/working/accel_train.py

In [ ]:
print("=" * 60)
print("Run 2: FSDP backend (same Python script!)")
print("=" * 60)
!accelerate launch --config_file /kaggle/working/accel_fsdp.yaml /kaggle/working/accel_train.py

## Part 3: Fine-Tune a Real Model — GPT-2 on Text Classification

Everything up to now used random data. Now we use a real pretrained model and real data.

In [ ]:
%%writefile /kaggle/working/finetune_gpt2.py
"""
Fine-tune GPT-2 for sentiment classification on SST-2.

Launch:
  accelerate launch --config_file accel_ddp.yaml finetune_gpt2.py
  accelerate launch --config_file accel_fsdp.yaml finetune_gpt2.py
"""
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer, AutoConfig
from datasets import load_dataset
from accelerate import Accelerator
from accelerate.utils import set_seed

BATCH_SIZE = 16
MAX_LEN = 128
LR = 2e-5
NUM_EPOCHS = 2
SEED = 42
MAX_TRAIN_SAMPLES = 1000  # Use a subset to keep it fast on Kaggle
MAX_EVAL_SAMPLES = 200


def main():
    set_seed(SEED)
    accelerator = Accelerator()
    accelerator.print(f"Backend: {accelerator.distributed_type} | Mixed precision: {accelerator.mixed_precision}")

    # ── Load tokenizer and model ───────────────────────────────────────────────
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

    model = GPT2ForSequenceClassification.from_pretrained(
        "gpt2",
        num_labels=2,
        pad_token_id=tokenizer.eos_token_id,
    )

    if accelerator.is_main_process:
        params = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"GPT-2 params: {params:,} | Trainable: {trainable:,}")

    # ── Load SST-2 (sentiment classification) ─────────────────────────────────
    # Only rank 0 downloads; others wait
    with accelerator.main_process_first():
        raw_dataset = load_dataset("glue", "sst2")

    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
        )

    with accelerator.main_process_first():
        tokenized = raw_dataset.map(tokenize, batched=True, remove_columns=["sentence", "idx"])
        tokenized = tokenized.rename_column("label", "labels")
        tokenized.set_format("torch")

    train_data = tokenized["train"].select(range(MAX_TRAIN_SAMPLES))
    eval_data  = tokenized["validation"].select(range(MAX_EVAL_SAMPLES))

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    eval_loader  = DataLoader(eval_data,  batch_size=BATCH_SIZE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    # ── Prepare — handles DDP/FSDP/DeepSpeed wrapping ─────────────────────────
    model, optimizer, train_loader, eval_loader = accelerator.prepare(
        model, optimizer, train_loader, eval_loader
    )

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        throughputs = []

        for batch in train_loader:
            t0 = time.perf_counter()
            optimizer.zero_grad()

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss
            accelerator.backward(loss)
            
            # Gradient clipping — important for transformer fine-tuning
            accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()

            elapsed = time.perf_counter() - t0
            throughputs.append(BATCH_SIZE * accelerator.num_processes / elapsed)
            epoch_loss += loss.item()

        # ── Evaluation ────────────────────────────────────────────────────────
        model.eval()
        correct = 0
        total = 0

        for batch in eval_loader:
            with torch.no_grad():
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"]
                )
            preds = outputs.logits.argmax(dim=-1)
            
            # Gather predictions from all processes
            preds, labels = accelerator.gather_for_metrics(
                (preds, batch["labels"])
            )
            correct += (preds == labels).sum().item()
            total += labels.numel()

        if accelerator.is_main_process:
            avg_loss = epoch_loss / len(train_loader)
            avg_tp = sum(throughputs) / len(throughputs)
            accuracy = correct / total * 100
            mem_0 = torch.cuda.memory_allocated(0) / 1e9
            print(
                f"[Epoch {epoch}] "
                f"loss={avg_loss:.4f} | "
                f"acc={accuracy:.1f}% | "
                f"{avg_tp:.0f} samples/s | "
                f"GPU0={mem_0:.1f}GB"
            )

    # ── Save ──────────────────────────────────────────────────────────────────
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        unwrapped = accelerator.unwrap_model(model)
        unwrapped.save_pretrained(
            "/kaggle/working/gpt2_sst2",
            save_function=accelerator.save
        )
        tokenizer.save_pretrained("/kaggle/working/gpt2_sst2")
        print("Model saved to /kaggle/working/gpt2_sst2")


if __name__ == "__main__":
    main()

In [ ]:
print("=" * 60)
print("Fine-tuning GPT-2 with DDP + fp16")
print("=" * 60)
!accelerate launch --config_file /kaggle/working/accel_ddp.yaml /kaggle/working/finetune_gpt2.py

In [ ]:
# Load the saved model and run inference
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer
import torch

model = GPT2ForSequenceClassification.from_pretrained("/kaggle/working/gpt2_sst2")
tokenizer = GPT2Tokenizer.from_pretrained("/kaggle/working/gpt2_sst2")
model.eval()

test_sentences = [
    "This movie was absolutely fantastic!",
    "I fell asleep halfway through, complete waste of time.",
    "The acting was decent but the plot was confusing.",
    "One of the best films I have seen in years.",
]

for sentence in test_sentences:
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax().item()
    confidence = torch.softmax(logits, dim=-1).max().item()
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    print(f"[{label} {confidence:.0%}] {sentence}")

## Part 4: Mixed Precision Deep Dive

Understand what fp16 does and why it matters.

In [ ]:
import torch
import torch.nn as nn
import time

D_MODEL = 1024
BATCH = 16
SEQ = 128

layer = nn.TransformerEncoderLayer(D_MODEL, nhead=16, dim_feedforward=D_MODEL*4, batch_first=True).to('cuda:0')
x = torch.randn(BATCH, SEQ, D_MODEL).to('cuda:0')

# Warm up
with torch.no_grad():
    _ = layer(x)

# ── FP32 (baseline) ───────────────────────────────────────────────────────────
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(20):
    with torch.no_grad():
        out_fp32 = layer(x)
torch.cuda.synchronize()
fp32_time = (time.perf_counter() - t0) / 20 * 1000

# ── FP16 with autocast ────────────────────────────────────────────────────────
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(20):
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
        out_fp16 = layer(x)
torch.cuda.synchronize()
fp16_time = (time.perf_counter() - t0) / 20 * 1000

print(f"fp32: {fp32_time:.2f} ms/step")
print(f"fp16: {fp16_time:.2f} ms/step")
print(f"Speedup: {fp32_time/fp16_time:.2f}x")
print()
print(f"Output dtype (fp32): {out_fp32.dtype}")
print(f"Output dtype (fp16): {out_fp16.dtype}")

# Check numerical difference
with torch.autocast(device_type='cuda', dtype=torch.float16):
    out_fp16_for_compare = layer(x)
diff = (out_fp32 - out_fp16_for_compare.float()).abs().mean().item()
print(f"\nMean absolute difference (fp32 vs fp16): {diff:.6f}")
print("Small difference is expected and acceptable.")

## Checkpoint Questions

1. What does `accelerator.prepare()` actually do when you run with the DDP config?
2. Why do we use `accelerator.backward(loss)` instead of `loss.backward()`?
3. What does `accelerator.gather_for_metrics()` do and why is it needed for evaluation?
4. What is `accelerator.unwrap_model()` and when do you need it?

**Answers:**
1. It wraps the model with DDP, adds DistributedSampler to the dataloader, moves the model to the correct device, and sets up mixed precision (GradScaler for fp16).
2. For fp16 training, `accelerator.backward()` calls `scaler.scale(loss).backward()` to prevent gradient underflow. For DeepSpeed, it calls `model_engine.backward()`. Using plain `loss.backward()` breaks fp16 and DeepSpeed.
3. `gather_for_metrics()` collects predictions from all processes onto rank 0 so you can compute global accuracy. Without it, each process would only compute accuracy on its own shard of the eval set.
4. `unwrap_model()` strips the DDP/FSDP/DeepSpeed wrapper to get the underlying `nn.Module`. Needed when saving, since the wrapper's state_dict has a different key format than the original model.

---

## Congratulations — You Have Completed All 5 Modules!

**What you have learned:**
- Module 00: Profiling, hardware basics, memory analysis
- Module 01: DDP — gradient synchronization, DistributedSampler, gradient accumulation
- Module 02: Model parallelism — layer splitting, the bubble problem
- Module 03: Pipeline parallelism — micro-batches, gradient checkpointing
- Module 04: ZeRO / FSDP / DeepSpeed — sharding optimizer, gradients, and params
- Module 05: Accelerate — one script for all backends, real model fine-tuning

**What to study next:**
- Tensor parallelism (Megatron-LM)
- Flash Attention 2
- `torch.compile` (PyTorch 2.0)
- Multi-node training on cloud VMs